# Spam or ham?

Can we distinguish between spam and ham using text embedding and a standard classifier?

## Get the text

The training data are in `training.pkl` and the test data are in `test.pkl`.  Each file contains a dictionary with two keys, `text` and `labels`.  The `text` value is a list of strings; each is either spam or ham.  The `labels` value is a list of class labels, 0 for ham and 1 for spam.

In [3]:
# The following controls whether embeddings are computed.
compute_embeddings = True

## What is to be done

* Embed the text using
  1. a commercial API;
  2. a locally hosted model.
* For each embedding, build and evaluate a standard classifier of your choice for distinguishing spam and ham.
* For each embedding, build and evaluate a cosine similarity classifier for distinguishing spam and ham.
* Choose a sentence,
  1. Make it a statement that ends with a period: `.`
  2. Take the same statement and make into a question ending with a question mark: `?`
  3. Embed both the statement and question.  Measure how similar or dissimilar the embeddings are.
* What to submit:
  1. Your code.
  2. A write-up that includes the following:
     * The name of the commercial API you used.
     * The name of the locally hosted model you used.
     * How well your classifiers did.
     * The results of your statement/question embedding experiment.
     * Your thoughts about how well the embeddings captured the semantic/syntactic differences between spam and ham?

I have included some code in this notebook that might help so you can focus on the experiments rather than having to figure out all the LLM tooling.

# Embed the text using a commercial API

All the commercial LLM services I looked into have an embedding endpoint.  If you go with a paid service, look into the cost of using the API, which tend to charge by the token, instead of a monthly subscription service.

## Embedding using Cohere's API

Cohere provides [a number of LLM endpoints](https://docs.cohere.com/docs/rate-limits) for various purposes, including embedding.

In [1]:
import cohere
import numpy as np
def embed_using_cohere(train, test):
    with open('cohere.txt', 'r') as f:
        api_key = f.read()
    api_key = api_key.replace('\n', '')
    co = cohere.Client(api_key)

    response = co.embed(
        texts=train["text"],
        input_type="classification",
        model="embed-v4.0"
    ).embeddings
    train["embeddings"] = np.array(response)

    response = co.embed(
        texts=test["text"],
        input_type="classification",
        model="embed-v4.0"
    ).embeddings
    test["embeddings"] = np.array(response)

## Embedding using Google's Gemini

Google also has an [embedding API](https://ai.google.dev/gemini-api/docs/embeddings).

In [ ]:
# Fixed: removed trailing embed_using_gemini(train, test) call; train/test are not loaded at this point
from google import genai

def embed_using_gemini(train, test):
    with open('gemini.txt', 'r') as f:
        api_key = f.read()
    api_key = api_key.replace('\n', '')

    client = genai.Client(api_key=api_key)

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=train["text"]
    )
    train["embeddings"] = np.array(result.embeddings)

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=test["text"]
    )
    test["embeddings"] = np.array(result.embeddings)

## Embedding using Mistral

Mistral also offers an [embedding option](https://docs.mistral.ai/capabilities/embeddings/text_embeddings) in their API.

In [ ]:
import os
from mistralai.client import Mistral

api_key = os.environ["MISTRAL_API_KEY"]
model = "mistral-embed"

client = Mistral(api_key=api_key)

embeddings_batch_response = client.embeddings.create(
    model=model,
    inputs=["Embed this sentence.", "As well as this one."],
)

## Perform the embedding with Cohere

In [ ]:
import pickle
import numpy as np

import os, shutil
os.makedirs("spam", exist_ok=True)
shutil.copy("train.pkl", "spam/train.pkl")
shutil.copy("test.pkl", "spam/test.pkl")

if compute_embeddings:
    with open("spam/train.pkl", "rb") as fileobj:
        cohere_train = pickle.load(fileobj)
    with open("spam/test.pkl", "rb") as fileobj:
        cohere_test = pickle.load(fileobj)
    embed_using_cohere(cohere_train, cohere_test)
    cohere_train["labels"] = np.array(cohere_train["labels"])
    cohere_test["labels"] = np.array(cohere_test["labels"])
    print(cohere_train["embeddings"].shape, cohere_test["embeddings"].shape)
    with open("spam/cohere_train_embeddings.pkl", "wb") as fileobj:
        pickle.dump(cohere_train, fileobj)
    with open("spam/cohere_test_embeddings.pkl", "wb") as fileobj:
        pickle.dump(cohere_test, fileobj)
else:
    with open("spam/cohere_train_embeddings.pkl", "rb") as fileobj:
        cohere_train = pickle.load(fileobj)
        cohere_train["embeddings"] = np.array(cohere_train["embeddings"])
        cohere_train["labels"] = np.array(cohere_train["labels"])
    with open("spam/cohere_test_embeddings.pkl", "rb") as fileobj:
        cohere_test = pickle.load(fileobj)
        cohere_test["embeddings"] = np.array(cohere_test["embeddings"])
        cohere_test["labels"] = np.array(cohere_test["labels"])

(100, 1536) (100, 1536)


# Embed the text using a locally hosted model

The [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) sentence transformer is a small model (22.7M parameters) that still performs pretty well.

In [ ]:
with open("huggingface.txt", "r") as f:
    hftoken = f.read()
hftoken = hftoken.replace('\n', '')

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

model_name = 'all-MiniLM-L6-v2'
local_model = SentenceTransformer(model_name, trust_remote_code=True, token=hftoken)

C:\Users\peyto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\peyto\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

In [ ]:
if compute_embeddings:
    with open("spam/train.pkl", "rb") as fileobj:
        local_train = pickle.load(fileobj)
    with open("spam/test.pkl", "rb") as fileobj:
        local_test = pickle.load(fileobj)
    local_train["embeddings"] = local_model.encode(local_train["text"], normalize_embeddings=True)
    local_test["embeddings"] = local_model.encode(local_test["text"], normalize_embeddings=True)
    local_train["labels"] = np.array(local_train["labels"])
    local_test["labels"] = np.array(local_test["labels"])
    print(local_train["embeddings"].shape, local_test["embeddings"].shape)
    with open("spam/local_train_embeddings.pkl", "wb") as fileobj:
        pickle.dump(local_train, fileobj)
    with open("spam/local_test_embeddings.pkl", "wb") as fileobj:
        pickle.dump(local_test, fileobj)
else:
    with open("spam/local_train_embeddings.pkl", "rb") as fileobj:
        local_train = pickle.load(fileobj)
        local_train["embeddings"] = np.array(local_train["embeddings"])
        local_train["labels"] = np.array(local_train["labels"])
    with open("spam/local_test_embeddings.pkl", "rb") as fileobj:
        local_test = pickle.load(fileobj)
        local_test["embeddings"] = np.array(local_test["embeddings"])
        local_test["labels"] = np.array(local_test["labels"])

(100, 384) (100, 384)


# Classifiers Using Cohere Embeddings

## Logistic Regression Classifier - Cohere

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf_cohere = LogisticRegression(random_state=42).fit(cohere_train["embeddings"], cohere_train["labels"])
y_pred_cohere = clf_cohere.predict(cohere_test["embeddings"])
train_acc = clf_cohere.score(cohere_train["embeddings"], cohere_train["labels"])
test_acc = clf_cohere.score(cohere_test["embeddings"], cohere_test["labels"])
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(classification_report(cohere_test["labels"], y_pred_cohere, target_names=["ham", "spam"]))

Train accuracy: 1.0000
Test accuracy:  0.9900
              precision    recall  f1-score   support

         ham       1.00      0.98      0.99        47
        spam       0.98      1.00      0.99        53

    accuracy                           0.99       100
   macro avg       0.99      0.99      0.99       100
weighted avg       0.99      0.99      0.99       100



## Cosine Similarity Classifier - Cohere

In [ ]:
from math import pi
from sklearn.metrics.pairwise import cosine_similarity

ham_rows = np.array(1 - cohere_train["labels"], dtype=bool)
mean_ham = np.mean(cohere_train["embeddings"][ham_rows], axis=0).reshape(1, -1)

spam_rows = np.array(cohere_train["labels"], dtype=bool)
mean_spam = np.mean(cohere_train["embeddings"][spam_rows], axis=0).reshape(1, -1)

angle = (180 / pi) * np.arccos(cosine_similarity(mean_ham, mean_spam))
print(f"Angle between mean ham and spam embeddings: {angle[0][0]:.2f} degrees")

X = np.vstack([mean_ham, mean_spam])
cos_theta = cosine_similarity(X, cohere_test["embeddings"])
y_pred_cos_cohere = np.argmax(cos_theta, axis=0)

n_wrong = np.sum(y_pred_cos_cohere != cohere_test["labels"])
cos_acc = np.mean(y_pred_cos_cohere == cohere_test["labels"])
print(f"Number misclassified: {n_wrong}")
print(f"Test accuracy: {cos_acc:.4f}")
print(classification_report(cohere_test["labels"], y_pred_cos_cohere, target_names=["ham", "spam"]))

Angle between mean ham and spam embeddings: 67.03 degrees
Number misclassified: 1
Test accuracy: 0.9900
              precision    recall  f1-score   support

         ham       1.00      0.98      0.99        47
        spam       0.98      1.00      0.99        53

    accuracy                           0.99       100
   macro avg       0.99      0.99      0.99       100
weighted avg       0.99      0.99      0.99       100



# Classifiers Using Local Model Embeddings

## Logistic Regression Classifier - Local Model

In [ ]:
clf_local = LogisticRegression(random_state=42).fit(local_train["embeddings"], local_train["labels"])
y_pred_local = clf_local.predict(local_test["embeddings"])
train_acc = clf_local.score(local_train["embeddings"], local_train["labels"])
test_acc = clf_local.score(local_test["embeddings"], local_test["labels"])
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(classification_report(local_test["labels"], y_pred_local, target_names=["ham", "spam"]))

Train accuracy: 1.0000
Test accuracy:  0.9800
              precision    recall  f1-score   support

         ham       1.00      0.96      0.98        47
        spam       0.96      1.00      0.98        53

    accuracy                           0.98       100
   macro avg       0.98      0.98      0.98       100
weighted avg       0.98      0.98      0.98       100



## Cosine Similarity Classifier - Local Model

In [ ]:
ham_rows = np.array(1 - local_train["labels"], dtype=bool)
mean_ham = np.mean(local_train["embeddings"][ham_rows], axis=0).reshape(1, -1)

spam_rows = np.array(local_train["labels"], dtype=bool)
mean_spam = np.mean(local_train["embeddings"][spam_rows], axis=0).reshape(1, -1)

angle = (180 / pi) * np.arccos(cosine_similarity(mean_ham, mean_spam))
print(f"Angle between mean ham and spam embeddings: {angle[0][0]:.2f} degrees")

X = np.vstack([mean_ham, mean_spam])
cos_theta = cosine_similarity(X, local_test["embeddings"])
y_pred_cos_local = np.argmax(cos_theta, axis=0)

n_wrong = np.sum(y_pred_cos_local != local_test["labels"])
cos_acc = np.mean(y_pred_cos_local == local_test["labels"])
print(f"Number misclassified: {n_wrong}")
print(f"Test accuracy: {cos_acc:.4f}")
print(classification_report(local_test["labels"], y_pred_cos_local, target_names=["ham", "spam"]))

Angle between mean ham and spam embeddings: 80.76 degrees
Number misclassified: 2
Test accuracy: 0.9800
              precision    recall  f1-score   support

         ham       1.00      0.96      0.98        47
        spam       0.96      1.00      0.98        53

    accuracy                           0.98       100
   macro avg       0.98      0.98      0.98       100
weighted avg       0.98      0.98      0.98       100



# Statement vs question

Make the same sentence both a statement and a question.  What is the angle between the embeddings of each?

In [ ]:
stmt = "I am the greatest computer scientist alive."
ques = "Am I the greatest computer scientist alive?"

In [ ]:
stmt_em = local_model.encode([stmt], normalize_embeddings=True)
ques_em = local_model.encode([ques], normalize_embeddings=True)
angle = (180 / pi) * np.arccos(cosine_similarity(stmt_em, ques_em))
print(f"Statement: {stmt}")
print(f"Question:  {ques}")
print(f"Angle between embeddings: {angle[0][0]:.2f} degrees")

Statement: I am the greatest computer scientist alive.
Question:  Am I the greatest computer scientist alive?
Angle between embeddings: 24.09 degrees


# Write-up

## Commercial API

The commercial API I used was **Cohere** with the `embed-v4.0` model. 

## Locally Hosted Model

The locally hosted model was **`all-MiniLM-L6-v2`** with its sentence transformer. 

## Classifier Results


| Embedding | Classifier | Train Acc | Test Acc |
|-----------|-----------|-----------|----------|
| Cohere | Logistic Regression |100% |99% |
| Cohere | Cosine Similarity |-- |99%  |
| Local (MiniLM) | Logistic Regression |100% |98% |
| Local (MiniLM) | Cosine Similarity | -- |98% |

## Statement vs. Question Experiment

- **Statement:** *I am the greatest computer scientist alive.*
- **Question:** *Am I the greatest computer scientist alive?*
- **Angle between embeddings:** 24.09 degrees


## Thoughts on Embeddings
Both embeddings worked very well at this task. Cohere did slightly better with 99% rather than the local Models 98% showing that the larger cloud based model is slightly more effective at this task compared to the smaller local model. It is also important to note that within the models there was no difference between logistic regression and cosine similarity classifiers. The models both captured the semantic and syntatic content well. Because the model places the question and statement so close together in the question experiment, we can see that the local model focuses heavily on the semantic content more than the syntatic form.

## Citations
Claude assisted me in multiple sections of the code development, helped me further understand some project concepts and helped me with debugging. 


